In [118]:
import chess
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import torch.optim as optim

def board_to_array(board):
    # Create an 19 x 8 x 8 array
    x = np.zeros((19, 8, 8), dtype=np.uint8)

    # Layers 0-11: Piece positions
    # White: pawn, knight, bishop, rook, queen, king
    # Black: pawn, knight, bishop, rook, queen, king
    piece_layer = {
        (chess.WHITE, chess.PAWN): 0,
        (chess.WHITE, chess.KNIGHT): 1,
        (chess.WHITE, chess.BISHOP): 2,
        (chess.WHITE, chess.ROOK): 3,
        (chess.WHITE, chess.QUEEN): 4,
        (chess.WHITE, chess.KING): 5,
        (chess.BLACK, chess.PAWN): 6,
        (chess.BLACK, chess.KNIGHT): 7,
        (chess.BLACK, chess.BISHOP): 8,
        (chess.BLACK, chess.ROOK): 9,
        (chess.BLACK, chess.QUEEN): 10,
        (chess.BLACK, chess.KING): 11,
    }

    for square, piece in board.piece_map().items():
        row = 7 - chess.square_rank(square)
        col = chess.square_file(square)
        layer = piece_layer[(piece.color, piece.piece_type)]
        x[layer, row, col] = 1

    # Layers 12-15: Castling rights
    x[12, :, :] = int(board.has_kingside_castling_rights(chess.WHITE))
    x[13, :, :] = int(board.has_queenside_castling_rights(chess.WHITE))
    x[14, :, :] = int(board.has_kingside_castling_rights(chess.BLACK))
    x[15, :, :] = int(board.has_queenside_castling_rights(chess.BLACK))

    # Layer 16: Location of a pawn that can legally be
    # en passant
    if board.ep_square is not None:
        ep_square = board.ep_square
        row = 7 - chess.square_rank(ep_square)
        col = chess.square_file(ep_square)
        x[16, row, col] = 1

    # Layer 17: Side to move
    x[17, :, :] = int(board.turn == chess.WHITE)

    # Board shape
    x[18, :, :] = 1

    return x


In [119]:
def add_moves(moves):
    board = chess.Board()
    for move_san in moves:
        move = board.parse_san(move_san)
        board.push(move)

        x = board_to_array(board)
        X.append(x)

In [120]:
scores = []
X = []
i= 0
with open("../../data/regressor_games.jsonl", encoding="utf-8") as games:
    for game in games:
        i+=1
        if i >1:
            break
        game = game[game.find("1."):]
        game = game.replace("!","").replace("?","").replace("#","").replace("] }","").replace("{ [%eval","").replace("..","")
        game = game.split()
        del game[::3]
        del game[-2:]
        scores.extend(float(eval) for eval in game[1::2])
        moves = game[::2]
        add_moves(moves)

X = np.stack(X)
scores = np.stack(scores)

X = torch.tensor(X, dtype=torch.float32)
scores = torch.tensor(scores, dtype=torch.float32).reshape(-1, 1)

In [121]:
print(X.shape)
print(scores.shape)

torch.Size([55, 19, 8, 8])
torch.Size([55, 1])


In [122]:
class deepChess(nn.Module):
    def __init__(self):
        super(deepChess, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=19, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv2 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv3 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.conv4 = nn.Conv2d(in_channels=128, out_channels=128, kernel_size=3, stride=1, padding='same')
        self.fc1 = nn.Linear(in_features=128*8*8, out_features=1)

    def forward(self, a0):
        a1 = F.relu(self.conv1(a0))           
        a2 = F.relu(self.conv2(a1))   
        a3 = F.relu(self.conv3(a2))           
        a4 = F.relu(self.conv4(a3)) 
        flat = a4.view(a4.shape[0], 128*8*8)
        z5 = self.fc1(flat)  
        return z5

    
model = deepChess()
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [123]:


def train_model(model, optimizer,criterion, X, Y, epochs):
    for epoch in range(epochs):
        model.train()

        Ypred = model(X)
        loss =  criterion(Ypred, Y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if epoch % 100 == 0:
            print(loss.item())

In [124]:
print("X shape:", X.shape)
print("scores shape:", scores.shape)

Ypred = model(X)
print("Ypred shape:", Ypred.shape)

X shape: torch.Size([55, 19, 8, 8])
scores shape: torch.Size([55, 1])
Ypred shape: torch.Size([55, 1])


In [125]:
train_model(model, optimizer, criterion, X, scores, 1000)

17.948270797729492
12.091475486755371
5.910264015197754
2.782503128051758
0.7709307670593262
0.19411295652389526
0.05202403664588928
0.029176630079746246
0.01197883952409029
0.027012409642338753
